# NB06 — L3: verificación simbólica de Attractor Convergence

Capa **L3** del plan auto-ordenado (NB04 encontró `proof_audit: 0/17`
demostraciones formales — **L3 es donde tienen que venir las garantías**).

**Herramienta:** `l3_verify.py` — 8 checks con sympy 1.14.0 (versión
idéntica local y LXC114), anclados a labels de `main.tex`. Sin LLM,
sin aleatoriedad, orden fijo (R10). **z3 NO disponible** en este entorno
(verificado local + 114): cualquier claim z3-dependiente sería SKIP.

**Clases de veredicto (honestas, sin mezclar):**

| clase | significado |
|---|---|
| `PASS` | identidad algebraica/simbolica cierta **exactamente** (racionales/sympy) |
| `CONDITIONAL` | la afirmacion es **hipotesis del paper**, no derivable por calculo |
| `PASS_GRID` | verificado en **grilla deterministica finita** — consistente, NO es prueba |

## ① Los 8 checks (orden fijo, evidencia por check)

In [ ]:
import sys, json
sys.path.insert(0, ".")
import sympy
print("sympy", sympy.__version__)
try:
    import z3
    print("z3", "disponible")
except ImportError:
    print("z3: NO disponible (condicional — checks sympy-only)")

from l3_verify import run_all, summary
rows = run_all()
for r in rows:
    print(f"[{r['verdict']:12s}] {r['id']:24s} <- {r['source']}")
s = summary(rows)
print("\nresumen:", json.dumps(s["by_verdict"]), "| total:", s["total"])

sympy 1.14.0
z3: NO disponible (condicional — checks sympy-only)
[PASS        ] pf_block_spectrum        <- eq:pf_matrix,eq:pf_lyap
[PASS        ] lambda_j_degenerate      <- eq:lambda_max
[PASS        ] lyapunov_positive        <- eq:lyapunov
[CONDITIONAL ] convergence_conditional  <- eq:convergence
[PASS        ] landauer_identity        <- cor:lyaplandauer
[PASS        ] s5_min_semantics         <- eq:v,prop:s5_deterministic
[PASS        ] margin_frontier          <- eq:margin
[PASS_GRID   ] cone_convex_grid         <- thm:hybrid_invariance

resumen: {"PASS": 6, "CONDITIONAL": 1, "PASS_GRID": 1} | total: 8


### Lectura

- **6 PASS** — identidades exactas: bloque Perron–Frobenius (eigenvals
  racionales `{1, 99/100 ×3}`), espectro degenerado de `J=0.99I`,
  no-negatividad de `L` (4 terminos + particion de unidad), identidad
  Lyapunov–Landauer (la conversion nats↔bits cierra **exacta** con `ln2`),
  semantica del agregador `Min` de `eq:v`, geometria de cuerda +
  `eps_frontier=0.7746` de `eq:margin`.
- **1 CONDITIONAL** — `thm:convergence`: `E[ΔL] ≤ 0` es la **hipotesis**
  del teorema («if α..δ are set so that…»), no se deriva de la definicion
  de `L` sin la dinamica `f_0`. Declarado honestamente, no disfrazado.
- **1 PASS_GRID** — convexidad del cono de `thm:hybrid_invariance`
  verificada por grilla finita de midpoints (0 violaciones): consistente
  con el argumento (g concava para θ≥0), **no constituye prueba formal**.

Hallazgo: el paper **no tiene pruebas formales** (0/17) pero sus
**identidades algebraicas centrales si cierran exactamente** bajo
verificacion simbolica; lo que queda condicional es dinamica/hipotesis.

## ② Deep-dive: los dos checks mas sustantivos

In [ ]:
from l3_verify import run_all
rows = {r["id"]: r for r in run_all()}
for cid in ("pf_block_spectrum", "landauer_identity"):
    r = rows[cid]
    print(f"=== {r['id']}  [{r['source']}]  {r['verdict']} ===")
    print(f"    claim: {r['claim']}")
    for e in r["evidence"]:
        print(f"    - {e}")
    print()

=== pf_block_spectrum  [eq:pf_matrix,eq:pf_lyap]  PASS ===
    claim: lambda_PF=1, subdominante=0.99 (x3), gap=10^-2, ln(0.99)=-0.010050
    - eigs(M) p_eq uniforme = {1: 1, 99/100: 3} (exacto) -> True
    - fijo simbolico M p = p en simplex: True
    - gap = 1/100 = 10^-2
    - ln(99/100) = -0.010050336 vs paper -0.010050 -> match@6dp=True

=== landauer_identity  [cor:lyaplandauer]  PASS ===
    claim: Omega_diss = kT ln2 |dH| = kT |sum <ln|F'| >|
    - kT*ln2*(nats/ln2) - kT*nats = True
    - |lambda_max| = 0.010050336 nats/tick
    -   = 0.014499570 bits/tick (conversion exacta)
    - Delta H en la 1a igualdad es bits; la 2a es nats — el factor ln2 cierra exactamente la identidad



### Lectura deep-dive

- **pf_block_spectrum**: la relaxation `p ← 0.99p + 0.01 p_eq` es
  exactamente `M = 99/100·I + 1/100·p_eq 1^T` → eigenvals **racionales**
  `{1: 1, 99/100: 3}` (sympy sobre la instancia uniforme + punto fijo
  simbolico en todo el simplex). El gap `10^-2` y `ln(0.99) = -0.0100503`
  cierran a la precision declarada (6 dp) — la igualdad «exactamente el
  espectro medido» de `eq:pf_lyap ↔ eq:lambda_max` **sí es exacta**.
- **landauer_identity**: `kT·ln2·(nats/ln2) ≡ kT·nats` — el factor `ln2`
  del Landauer cierra la identidad **si y solo si** `ΔH` de la primera
  igualdad está en bits y la segunda en nats (conversion exacta). El paper
  no lo escribe explícitamente; la verificacion lo hace.

## Tests de la capa L3

In [ ]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "--version"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print("pytest no disponible en este env — suite 12/12 en local (R1 locality)")
else:
    r = subprocess.run(
        [sys.executable, "-m", "pytest", "test_l3_verify.py", "-q"],
        capture_output=True, text=True)
    print("\n".join((r.stdout + r.stderr).strip().splitlines()[-2:]))
    print("rc =", r.returncode, "(0 = 12/12 PASS)")

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
12 passed, 91 warnings in 12.05s
rc = 0 (0 = 12/12 PASS)


## Estado y siguiente

- **NB06 listo:** 8 checks simbolicos, **6 PASS + 1 CONDITIONAL +
  1 PASS_GRID**, 12/12 tests locales. Linea NB00→NB06 completa:
  situacion → B1 → B2/C → NB03 depth → NB04 L2-full → NB05 grafos →
  NB06 L3.
- **Hallazgo L3 para el paquete Fierro:** 0/17 proofs en el paper, pero
  6 identidades algebraicas clave **cierran exacto** bajo sympy; 1 es
  hipotesis por diseno (dinamica); 1 solo grilla finita.
- **Pendientes S5:** revision de `graph_epistemic_proposal.vsm`
  (D3/D4); envio del paquete a Fierro (addendum rev2 + carta + freeze
  v0.9) — **NADA enviado**; L4/L5 (experimentos reproducidos / refs
  externas) quedarian como NB07+ si rmw3 los ordena.